In [ ]:
# Cell 2: Imports
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import confusion_matrix, classification_report
import os

from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, Conv2D, MaxPooling2D, Dense, Dropout,
                                   BatchNormalization, GlobalAveragePooling2D,
                                   GaussianNoise, concatenate)
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam

import tensorflow as tf
print("TensorFlow version:", tf.__version__)



In [ ]:
# Cell 3: GPU Configuration & Dataset Check
tf.keras.backend.clear_session()

# Configure GPU
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("GPU memory growth configured")
    except RuntimeError as e:
        print(e)

# dataset structure
def check_dataset_structure():
    train_path = "/train"
    val_path = "/val"
    test_path = "/test"

    for split_path, split_name in [(train_path, 'TRAIN'), (val_path, 'VAL'), (test_path, 'TEST')]:
        print(f"\n{split_name} SET:")
        if os.path.exists(split_path):
            for class_name in os.listdir(split_path):
                class_path = os.path.join(split_path, class_name)
                if os.path.isdir(class_path):
                    num_images = len([f for f in os.listdir(class_path) if f.endswith(('.jpg', '.png', '.jpeg'))])
                    print(f"  {class_name}: {num_images} images")
        else:
            print(f"  Path not found: {split_path}")

check_dataset_structure()

In [ ]:
# Cell 4: Configuration
IMG_SIZE = (384, 384)  # Higher resolution for better artifact detection
BATCH_SIZE = 16        # Smaller batches for higher resolution

print("Configuration:")
print(f"Image Size: {IMG_SIZE}")
print(f"Batch Size: {BATCH_SIZE}")

In [ ]:
# STEP 2: IMPORT
import tensorflow as tf
from tensorflow import keras
import numpy as np
import os
from sklearn.utils.class_weight import compute_class_weight

In [ ]:
# STEP 3:
print("=== LOADING PHASE 2 MODEL (Where CG was 78%) ===")

model_path = '/phase2_cg_biased_model.keras'
model = tf.keras.models.load_model(model_path)



In [ ]:
# STEP 4: ENHANCED DATA GENERATORS WITH BALANCED PREPROCESSING
from tensorflow.keras.preprocessing.image import ImageDataGenerator


train_path = "/train"
val_path = "/val"
test_path = "/test"

print(f"Train path: {train_path}")
print(f"Val path: {val_path}")
print(f"Test path: {test_path}")

In [ ]:
# ENHANCED Data generators with balanced augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,  # Reduced from 20 for more stability
    width_shift_range=0.15,  # Reduced from 0.2
    height_shift_range=0.15,  # Reduced from 0.2
    horizontal_flip=True,
    brightness_range=[0.9, 1.1],  # Added for robustness
    zoom_range=0.1,  # Mild zoom
    fill_mode='nearest'
)

val_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

# Train generator
train_generator = train_datagen.flow_from_directory(
    directory=train_path,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    shuffle=True
)

# Validation generator
val_generator = val_datagen.flow_from_directory(
    directory=val_path,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    shuffle=False
)

# Test generator
test_generator = test_datagen.flow_from_directory(
    directory=test_path,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    shuffle=False
)

print("✅ Data generators ready!")
class_names = list(train_generator.class_indices.keys())
print(f"Detected class order: {class_names}")

# STEP 5: SMART WEIGHT CALCULATION & ENHANCED CALLBACKS
print("\n=== SMART CLASS BALANCING SETUP ===")

# Calculate actual class distribution for smarter weights
class_counts = []
for class_name in class_names:
    class_path = os.path.join(train_path, class_name)
    count = len(os.listdir(class_path))
    class_counts.append(count)
    print(f"  {class_name}: {count} images")

total_samples = sum(class_counts)
print(f"Total training samples: {total_samples}")

# Calculate balanced weights automatically
balanced_weights = {}
for i, count in enumerate(class_counts):
    balanced_weights[i] = total_samples / (len(class_counts) * count)

print("Automatically calculated balanced weights:", balanced_weights)

final_weights = {
    0: min(1.15, balanced_weights[0]),
    1: max(0.9, balanced_weights[1]),
    2: min(1.15, balanced_weights[2])
}

print(f"Final safe weights: {final_weights}")

# ENHANCED CALLBACKS for stability
enhanced_callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=3,  # Increased patience
        restore_best_weights=True,
        min_delta=0.002,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2,
        min_lr=1e-7,
        verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        '/content/drive/MyDrive/phase3_best_checkpoint.keras',
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    ),
    tf.keras.callbacks.CSVLogger(
        '/content/drive/MyDrive/phase3_training_log.csv',
        separator=',',
        append=False
    )
]










In [ ]:
# STEP 6: WITH LOWER LEARNING RATE FOR SAFE BALANCING
print("\n=== COMPILING WITH SAFE SETTINGS ===")

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),  # Reduced LR
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("Using lower learning rate (1e-5) for stable balancing")

In [ ]:

import tensorflow as tf
from tensorflow import keras
import numpy as np
import os

print("✅ Libraries imported!")

print("=== LOADING PHASE 2 MODEL ===")
model_path = '/phase2_cg_biased_model.keras'
model = tf.keras.models.load_model(model_path)
print(f"✅ Phase 2 model loaded")


from tensorflow.keras.preprocessing.image import ImageDataGenerator


train_path = "/content"
val_path = "/content"
test_path = "/content"

IMG_HEIGHT, IMG_WIDTH = 384, 384


train_datagen = ImageDataGenerator(
    rescale=1./255,
    horizontal_flip=True,
    brightness_range=[0.9, 1.1]
)

val_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    directory=train_path,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=12,  # Optimized for speed/performance balance
    class_mode='categorical',
    shuffle=True
)

val_generator = val_datagen.flow_from_directory(
    directory=val_path,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=12,
    class_mode='categorical',
    shuffle=False
)

test_generator = test_datagen.flow_from_directory(
    directory=test_path,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=12,
    class_mode='categorical',
    shuffle=False
)

print("✅ Optimized data generators ready!")
class_names = list(train_generator.class_indices.keys())
print(f"Class order: {class_names}")


final_weights = {0: 1.1, 1: 0.95, 2: 1.1}
print(f"Using proven weights: {final_weights}")

# STEP 6: OPTIMIZED TRAINING SETTINGS
print("\n=== PHASE 3: OPTIMIZED BALANCING ===")

optimized_callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=2,
        restore_best_weights=True,
        min_delta=0.005
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=1,
        min_lr=1e-7
    )
]


model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=2e-5),  # Slightly higher
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("🎯 TARGET: All classes >65% for fine-tuning preparation")
print("⏱️  Expected time: 15-20 min per epoch")


phase3_history = model.fit(
    train_generator,
    epochs=4,  # Reduced from 6
    validation_data=val_generator,
    class_weight=final_weights,
    callbacks=optimized_callbacks,
    verbose=1,
    steps_per_epoch=len(train_generator) // 2
)

print("✅ Phase 3 balancing completed!")



In [ ]:
print("=== TESTING BEST MODEL (Epoch 3) ===")

test_loss, test_accuracy = model.evaluate(test_generator)
print(f"🎯 Test Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")


from sklearn.metrics import classification_report

test_generator.reset()
predictions = model.predict(test_generator)
y_pred = np.argmax(predictions, axis=1)
y_true = test_generator.classes

print("\n📊 CLASS-WISE PERFORMANCE:")
print(classification_report(y_true, y_pred, target_names=class_names))

In [ ]:
print("=== PHASE 3.5: TARGETED GRAPHICS BOOST ===")

targeted_weights = {
    0: 1.0,   # GAN: Neutral (already good at 76%)
    1: 1.3,   # Graphics: +30% boost (fix the 48% recall)
    2: 1.0    # Real: Neutral (decent at 64%)
}

print(f"Targeted weights: {targeted_weights}")
print("Strategy: Boost Graphics while maintaining GAN/Real")

# SHORT focused training (2-3 epochs only)
focused_callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=2,
        restore_best_weights=True,
        min_delta=0.003
    )
]

phase3_5_history = model.fit(
    train_generator,
    epochs=3,
    validation_data=val_generator,
    class_weight=targeted_weights,
    callbacks=focused_callbacks,
    verbose=1
)

print("✅ Graphics boost completed!")

In [ ]:
print("=== VERIFYING GRAPHICS BOOST ===")

# Quick test on validation set
val_loss, val_accuracy = model.evaluate(val_generator, verbose=0)
print(f"Val Accuracy: {val_accuracy:.4f} ({val_accuracy*100:.2f}%)")


from sklearn.metrics import classification_report

val_generator.reset()
predictions = model.predict(val_generator, verbose=0)
y_pred = np.argmax(predictions, axis=1)
y_true = val_generator.classes

print("\n📊 VALIDATION SET PERFORMANCE:")
print(classification_report(y_true, y_pred, target_names=class_names))


if val_accuracy >= 0.66:  # If better than previous 66.48%
    model.save('/content')
    print("✅ Graphics boost successful - model saved!")
else:
    print("⚠️  Graphics boost needs adjustment")

In [ ]:
model.trainable = True

# Unfreeze only 60 layers total (moderate)
for layer in model.layers[:-60]:
    layer.trainable = False

print(f"Unfrozen layers: {len([l for l in model.layers if l.trainable])}")

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=2e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

quick_callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=2,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ModelCheckpoint(
        '/content',
        monitor='val_accuracy',
        save_best_only=True
    )
]


quick_history = model.fit(
    train_generator,
    epochs=6,
    validation_data=val_generator,
    class_weight=None,
    callbacks=quick_callbacks,
    verbose=1
)

In [ ]:
print("=== FINAL PERFORMANCE CHECK ===")
test_loss, test_accuracy = model.evaluate(test_generator)
print(f"🎯 FINAL TEST ACCURACY: {test_accuracy*100:.2f}%")

# Class-wise performance
test_generator.reset()
predictions = model.predict(test_generator)
y_pred = np.argmax(predictions, axis=1)
y_true = test_generator.classes

print("\n📊 FINAL CLASS PERFORMANCE:")
print(classification_report(y_true, y_pred, target_names=class_names))

In [ ]:
print("=== FINAL AGGRESSIVE PUSH FOR 80% ===")

# STRATEGY: UNFREEZE EVERYTHING + FOCAL LOSS
model.trainable = True

# Unfreeze ALL layers (maximum learning capacity)
for layer in model.layers:
    if not isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = True

print("🔥 UNFROZE ALL LAYERS!")

# FOCAL LOSS for class imbalance
def focal_loss(gamma=2.0, alpha=0.25):
    def focal_loss_fixed(y_true, y_pred):
        # Focal loss implementation
        epsilon = tf.keras.backend.epsilon()
        y_pred = tf.clip_by_value(y_pred, epsilon, 1. - epsilon)
        cross_entropy = -y_true * tf.math.log(y_pred)
        weight = alpha * y_true * tf.pow((1 - y_pred), gamma)
        focal_loss_value = weight * cross_entropy
        return tf.reduce_mean(focal_loss_value, axis=-1)
    return focal_loss_fixed

# AGGRESSIVE COMPILE
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=5e-5),
    loss=focal_loss(),
    metrics=['accuracy']
)

print("🎯 FINAL TARGET: 70-75% in 1 hour")
print("USING: Full unfreezing + Focal Loss")

# FINAL QUICK TRAINING
final_history = model.fit(
    train_generator,
    epochs=8,
    validation_data=val_generator,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(patience=2)
    ],
    verbose=1
)

print("✅ FINAL AGGRESSIVE TRAINING COMPLETED!")

In [ ]:
print("=== FINAL RESULTS CHECK ===")

# Quick test
test_loss, test_accuracy = model.evaluate(test_generator, verbose=0)
print(f"🎯 FINAL TEST ACCURACY: {test_accuracy*100:.2f}%")

if test_accuracy >= 0.68:
    model.save('/content')
    print("✅ NEW BEST MODEL SAVED! 70% ACHIEVED!")
else:

    model.save('/content')
    print("✅ Improved model saved!")

